
# Introduction

This Notebook demonstrates how to fine-tune BERT using Transformer library for a sentiment analysis task.
The data used for fine-tuning is a financial sentiment analysis dataset.

# Imports and configurations

In [ ]:
!!pip install -q evaluate

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer
import numpy as np
import evaluate
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Data preparation

## Loading and preparing the dataset

If you are on Kaggle, just import the dataset and load it as a dataframe using Pandas.
Otherwise, download the file and stored it locally as a CSV file.

In [ ]:
# replace the path with "data.csv" if you are running the Notebook locally
file_path = "/kaggle/input/datasets/sbhatti/financial-sentiment-analysis/data.csv"
df = pd.read_csv(file_path)

We convert labes into numerical form:

In [ ]:
label_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df["label"] = df["Sentiment"].map(label_mapping)

Let's check for data unbalance.

In [ ]:
df["label"].value_counts()

Because we have a data inbalance not extreme, but relatively large, and because the overall data dimension is not very large, we will not perform downsampling, we will use class weights.

Let's split the dataset.

In [ ]:
# 95% train+validation, 5% test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.05,
    stratify=df["label"],
    random_state=42
)

# from the remaining 95%, keep validation around 10% of the remaining data
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    stratify=train_val_df["label"],
    random_state=42
)

## Tokenization using BERT

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

We define a tokenizing function.

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["Sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

## Create Hugging Face Dataset
To use the `Trainer` API efficiently, we convert the data into Hugging Face `Dataset` format:

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["Sentence", "label"]])
val_dataset = Dataset.from_pandas(val_df[["Sentence", "label"]])
test_dataset = Dataset.from_pandas(test_df[["Sentence", "label"]])

Apply tokenization.

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Set the format for PyTorch.

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# Prepare model

## Loading the pretraind BERT model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

## Define training configuration

Define function to compute metrics.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

Define the training arguments.

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-financial-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

Because the data is slightly unbalanced, we will use class weights.

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

# Train the model

Transformers uses a `Trainer` class for training.

```python
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
```

We would like to use the class weights. 

We create a custom weighted trainer for this.

In [ ]:
class_weights = torch.tensor(class_weights, dtype=torch.float)

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## Evaluation of the model

After training, we evaluate.

In [ ]:
results = trainer.evaluate()
print(results)

# Make predictions

In [ ]:
id_to_label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

def predict(text):
    model.eval()

    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = logits.argmax(dim=-1).item()

    return id_to_label[predicted_class]

In [ ]:
text = "The company reported strong earnings growth."
print(predict(text))

Note: during inference, the input tensors must be moved to the same device as the model. Otherwise, PyTorch raises a device mismatch error.

# Model validation and testing

In [ ]:
def plot_confusion_matrix_for_dataset(dataset, title):
    predictions_output = trainer.predict(dataset)

    y_true = predictions_output.label_ids
    y_pred = np.argmax(predictions_output.predictions, axis=-1)

    cm = confusion_matrix(y_true, y_pred)

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["negative", "neutral", "positive"]
    )

    display.plot(values_format="d")
    plt.title(title)
    plt.show()

## Validation set

In [ ]:
plot_confusion_matrix_for_dataset(val_dataset, "Validation Confusion Matrix")

## Testing set

In [ ]:
plot_confusion_matrix_for_dataset(test_dataset, "Test Confusion Matrix")